# Lab 2 — Spike experiments

Цель: проверить два рискованных допущения MVP, не выдавая предположения за факты.

**Spike A:** сможет ли модель одновременно соблюдать несколько ограничений и вернуть данные, проходящие программную проверку.

**Spike B:** помогает ли один корректирующий запрос после невалидного/нарушающего ограничения ответа.

Эксперимент рассчитан на GigaChat 2 Lite, выбранный в Lab 1. При отсутствии credentials notebook **не подставляет выдуманные результаты** и отмечает запуск как `not_run`.


In [1]:
!pip -q install gigachat pydantic pandas

import os, json, re, time
from pathlib import Path
import pandas as pd
from pydantic import BaseModel, Field

try:
    from google.colab import userdata
    GIGACHAT_CREDENTIALS = userdata.get("GIGACHAT_CREDENTIALS")
except Exception:
    GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")

print("Credentials configured:", bool(GIGACHAT_CREDENTIALS))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.8 MB/s eta 0:00:00
Credentials configured: True


In [2]:
class Ingredient(BaseModel):
    name: str
    amount_g: float = Field(ge=0)

class Meal(BaseModel):
    name: str
    meal_type: str
    ingredients: list[Ingredient]
    cook_time_min: float = Field(ge=0)

class DayPlan(BaseModel):
    day: int
    meals: list[Meal]

class MealPlan(BaseModel):
    days: list[DayPlan]
    notes: str = ""

def extract_json(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        if not match:
            raise
        return json.loads(match.group(0))

def constraint_report(plan, excluded, max_cook_time, pantry_names):
    ingredient_names = [
        i.name.lower()
        for d in plan.days for m in d.meals for i in m.ingredients
    ]
    forbidden_hits = [x for x in excluded if any(x.lower() in n for n in ingredient_names)]
    long_meals = [m.name for d in plan.days for m in d.meals if m.cook_time_min > max_cook_time]
    pantry_hits = [p for p in pantry_names if any(p.lower() in n for n in ingredient_names)]
    return {
        "forbidden_hits": forbidden_hits,
        "long_meals": long_meals,
        "pantry_hits": pantry_hits,
        "valid": not forbidden_hits and not long_meals
    }


## Spike A — несколько ограничений одновременно

**Допущение:** одного прямого prompt с несколькими ограничениями достаточно, чтобы модель сформировала структурированный ответ, который проходит программную проверку.

Тест одновременно задаёт цель, бюджет, запрет рыбы, pantry и лимит времени. Это не измеряет «полезность» меню, а только проверяемое соблюдение технических условий.


In [3]:
SYSTEM_PROMPT = """Ты — AI-помощник Планировщика питания.
Верни только JSON без Markdown в формате:
{"days":[{"day":1,"meals":[{"name":"...","meal_type":"breakfast|lunch|dinner|snack","ingredients":[{"name":"...","amount_g":100}],"cook_time_min":20}]}],"notes":"..."}
Не используй запрещённые продукты. Используй pantry. Не превышай max cook time. Не придумывай точную стоимость и КБЖУ.
"""

SPIKE_A = {
    "goal": "поддержание веса",
    "budget": "2500 RUB/week",
    "days": 3,
    "excluded": ["рыба", "арахис"],
    "pantry": ["гречка", "яйца", "курица"],
    "max_cook_time": 30
}

user_a = f"Составь план на {SPIKE_A['days']} дня. Цель: {SPIKE_A['goal']}. Бюджет: {SPIKE_A['budget']}. Запрещено: {', '.join(SPIKE_A['excluded'])}. Дома: {', '.join(SPIKE_A['pantry'])}. Максимум приготовления: {SPIKE_A['max_cook_time']} минут."


In [4]:
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

client = GigaChat(credentials=GIGACHAT_CREDENTIALS, model="GigaChat-2", verify_ssl_certs=False) if GIGACHAT_CREDENTIALS else None

def call_model(system, user):
    if client is None:
        return {"status": "not_run", "reason": "GIGACHAT_CREDENTIALS not configured"}
    started = time.perf_counter()
    response = client.chat(Chat(messages=[
        Messages(role=MessagesRole.SYSTEM, content=system),
        Messages(role=MessagesRole.USER, content=user)
    ]))
    latency = time.perf_counter() - started
    text = response.choices[0].message.content
    return {"status": "ok", "latency_sec": latency, "text": text, "finish_reason": getattr(response.choices[0], "finish_reason", None)}

raw_a = call_model(SYSTEM_PROMPT, user_a)
print({k:v for k,v in raw_a.items() if k != 'text'})


{'status': 'ok', 'latency_sec': 5.704096864000007, 'finish_reason': 'stop'}


In [5]:
def evaluate_a(raw):
    if raw["status"] != "ok":
        return {"status": raw["status"], "json_valid": None, "schema_valid": None, "constraints_valid": None}
    out = {"status":"ok", "json_valid":False, "schema_valid":False, "constraints_valid":False}
    try:
        data = extract_json(raw["text"]); out["json_valid"] = True
        plan = MealPlan.model_validate(data); out["schema_valid"] = True
        rep = constraint_report(plan, SPIKE_A["excluded"], SPIKE_A["max_cook_time"], SPIKE_A["pantry"])
        out.update(rep)
        out["constraints_valid"] = rep["valid"]
    except Exception as e:
        out["error"] = str(e)[:300]
    return out

result_a = evaluate_a(raw_a)
print(result_a)


{'status': 'ok', 'json_valid': True, 'schema_valid': True, 'constraints_valid': True, 'forbidden_hits': [], 'long_meals': [], 'pantry_hits': ['гречка', 'яйца', 'курица'], 'valid': True}


## Spike B — корректирующий запрос

**Допущение:** если первичный ответ нарушает проверяемое ограничение или не проходит схему, один короткий corrective prompt сможет исправить ответ без построения полноценного агента.

Важно: эксперимент сравнивает первичный ответ и ответ после **одного** retry. Если первичный ответ уже валиден, это фиксируется как `not_needed`, а не как доказательство эффективности retry.


In [6]:
CORRECTION_SYSTEM = SYSTEM_PROMPT + "\nПри исправлении верни полный исправленный JSON, а не только изменённый фрагмент."

def correction_prompt(original_text, errors):
    return (
        "Исправь предыдущий ответ. Верни только полный JSON по исходной схеме. "
        "Исправь конкретные нарушения: " + json.dumps(errors, ensure_ascii=False) +
        "\nПредыдущий ответ:\n" + original_text
    )

def run_spike_b(raw):
    if raw["status"] != "ok":
        return {"status": raw["status"]}
    first = evaluate_a(raw)
    if first.get("constraints_valid") is True and first.get("schema_valid") is True:
        return {"status":"not_needed", "first":first, "retry":None}
    errors = {k:first.get(k) for k in ["forbidden_hits","long_meals"] if first.get(k)}
    retry_raw = call_model(CORRECTION_SYSTEM, correction_prompt(raw["text"], errors))
    second = evaluate_a(retry_raw) if retry_raw.get("status") == "ok" else {"status":retry_raw.get("status")}
    return {"status":"tested", "first":first, "retry":second, "retry_latency_sec":retry_raw.get("latency_sec")}

result_b = run_spike_b(raw_a)
print(json.dumps(result_b, ensure_ascii=False, indent=2))


{
  "status": "not_needed",
  "first": {
    "status": "ok",
    "json_valid": true,
    "schema_valid": true,
    "constraints_valid": true,
    "forbidden_hits": [],
    "long_meals": [],
    "pantry_hits": [
      "гречка",
      "яйца",
      "курица"
    ],
    "valid": true
  },
  "retry": null
}


## Итоговая фиксация результатов

Notebook не подставляет результаты заранее. После запуска в среде с credentials ниже формируется краткая таблица.

Интерпретация:
- `worked` — проверка прошла в заданных условиях;
- `did_not_work` — проверка не прошла;
- `worked_differently` — наблюдалось поведение, отличное от ожидаемого;
- `not_run` — эксперимент фактически не запускался и не используется как доказательство.


In [7]:
def classify_a(r):
    if r.get("status") == "not_run": return "not_run"
    if r.get("constraints_valid") is True and r.get("schema_valid") is True: return "worked"
    return "did_not_work"

print("Spike A verdict:", classify_a(result_a))

if result_b.get("status") == "not_needed":
    print("Spike B verdict: not_needed (первичный ответ уже прошёл проверки)")
elif result_b.get("status") == "tested":
    first_ok = result_b["first"].get("constraints_valid") is True
    retry_ok = result_b["retry"].get("constraints_valid") is True and result_b["retry"].get("schema_valid") is True
    print("Spike B verdict:", "worked" if (not first_ok and retry_ok) else "did_not_work")
else:
    print("Spike B verdict: not_run")


Spike A verdict: worked
Spike B verdict: not_needed (первичный ответ уже прошёл проверки)


In [8]:
# Сохраняем машинно-читаемый результат для отчёта.
# Метаданные latency/finish_reason берутся из фактического ответа API.
summary = {
    "spike_a": {**result_a, "latency_sec": raw_a.get("latency_sec"), "finish_reason": raw_a.get("finish_reason")},
    "spike_b": result_b,
    "note": "Results are populated only by an actual run; no synthetic model results are inserted."
}
Path("lab2_spike_results.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved lab2_spike_results.json")

Saved lab2_spike_results.json
